<a href="https://colab.research.google.com/github/crd10lop/CodingLinealOptimizacion/blob/main/Programaci%C3%B3n_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizacion Lineal: Simplex y Harmony Search

Aplicacion web interactiva para resolver problemas de programacion lineal usando dos enfoques: el metodo exacto **Simplex** y la metaheuristica **Harmony Search**. Incluye visualizaciones 2D y 3D con Plotly.

## Equipo

| Nombre | Correo |
|---|---|
| Juan Pablo Uribe González | juan.uribe4@udea.edu.co |
| Cristian David Diez López | cristian.diez@udea.edu.co |
| Zareth Mariana Vega Sánchez | zarethmariana.vega@udea.edu.co |

## Curso

**Asignatura:** Optimización  
**Docente:** Ronald Akerman Ortiz García  
**Departamento de Ingeniería de Sistemas**  
**Universidad de Antioquia — Semestre 2026-1**

## Contexto

Este proyecto hace parte de la actividad de programación lineal del curso de Optimización. El equipo tuvo asignado el algoritmo **Harmony Search** como método metaheurístico a estudiar e implementar. A partir de un notebook desarrollado en Google Colab, se construyó una aplicación web interactiva con Streamlit que permite comparar el método exacto Simplex (Gran M) con Harmony Search, incluyendo análisis de sensibilidad y visualizaciones 2D y 3D generadas con Plotly.

## Despliegue en Streamlit Community Cloud (gratuito)

Como la app esta pensada para verse sin instalar nada localmente, se publico con el siguiente enlace: 
**[codinglinealoptimizacion.streamlit.app](https://codinglinealoptimizacion.streamlit.app).**

In [ ]:

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

class InterfazPL:
    def __init__(self):
        # --- 1. CONFIGURACIÓN INICIAL ---
        self.lbl_titulo = widgets.HTML("<h3>Calculadora de Programación Lineal (Gran M + Múltiples Óptimos)</h3>")

        self.tipo_opt = widgets.Dropdown(options=['Maximizar', 'Minimizar'], description='Objetivo:', layout=widgets.Layout(width='200px'))

        # MEJORA 1: Control dinámico para los coeficientes de la F.O. (Variables)
        self.num_vars = widgets.BoundedIntText(value=2, min=2, max=10, description='Coeficientes Z (Variables):', style={'description_width': 'initial'}, layout=widgets.Layout(width='220px'))
        self.num_rest = widgets.BoundedIntText(value=3, min=1, max=10, description='Restricciones:', layout=widgets.Layout(width='150px'))

        self.btn_generar = widgets.Button(description='Generar Matriz', button_style='info')
        self.btn_generar.on_click(self.generar_campos)

        self.lbl_vars_info = widgets.HTML("<small><i>(Gráfico disponible solo para 2 variables)</i></small>", layout=widgets.Layout(margin='0 10px 0 5px'))

        self.config_box = widgets.HBox([self.tipo_opt, self.num_vars, self.num_rest, self.btn_generar])

        self.matriz_area = widgets.VBox()
        self.output_area = widgets.Output()

        display(self.lbl_titulo, self.config_box, self.lbl_vars_info, self.matriz_area, self.output_area)

    def generar_campos(self, b):
        self.matriz_area.children = []
        n_vars = self.num_vars.value
        n_rest = self.num_rest.value

        # --- 2. FUNCIÓN OBJETIVO (Z) ---
        lbl_z = widgets.HTML("<b>Función Objetivo:</b>")
        self.inputs_c = [widgets.FloatText(value=0, layout=widgets.Layout(width='70px')) for _ in range(n_vars)]

        elementos_z = []
        for i in range(n_vars):
            elementos_z.append(widgets.Label(f"C{i+1} (X{i+1}):"))
            elementos_z.append(self.inputs_c[i])
        fila_z = widgets.HBox(elementos_z)

        # --- 3. RESTRICCIONES (A y b) ---
        lbl_restr = widgets.HTML("<br><b>Restricciones estructurales:</b>")
        self.inputs_A = []
        self.inputs_signo = []
        self.inputs_b = []

        filas_ui = []
        for i in range(n_rest):
            fila_A = [widgets.FloatText(value=0, layout=widgets.Layout(width='70px')) for _ in range(n_vars)]
            signo = widgets.Dropdown(options=['<=', '>=', '='], value='<=', layout=widgets.Layout(width='60px'))
            rhs = widgets.FloatText(value=0, layout=widgets.Layout(width='70px'))

            self.inputs_A.append(fila_A)
            self.inputs_signo.append(signo)
            self.inputs_b.append(rhs)

            elementos_fila = [widgets.Label(f"R{i+1}: ")] + fila_A + [signo, rhs]
            filas_ui.append(widgets.HBox(elementos_fila))

        self.btn_resolver = widgets.Button(description='Resolver Problema', button_style='success', icon='check')
        self.btn_resolver.on_click(self.resolver_problema)

        self.matriz_area.children = [lbl_z, fila_z, lbl_restr] + filas_ui + [widgets.HTML("<br>"), self.btn_resolver]

    def resolver_problema(self, b):
        with self.output_area:
            clear_output(wait=True)
            print("⏳ Analizando modelo matemático...")

            es_max = (self.tipo_opt.value == 'Maximizar')
            vector_c = np.array([inp.value for inp in self.inputs_c])
            matriz_A = np.array([[inp.value for inp in fila] for fila in self.inputs_A])
            vector_signos = np.array([inp.value for inp in self.inputs_signo])
            vector_b = np.array([inp.value for inp in self.inputs_b])

            try:
                # --- RESOLUCIÓN SIMPLEX / GRAN M ---
                print("\n" + "="*55)
                print("📗 ALGORITMO SIMPLEX (GRAN M): EVOLUCIÓN TABULAR")
                print("="*55)

                c_calc = vector_c.copy()
                if not es_max:
                    c_calc = -c_calc # Estandarización a Max

                tablero_inicial, columnas, variables_base = self.init_tablero_completo(c_calc, matriz_A, vector_b, vector_signos)
                
                #SPRINT 3
                variables_base_iniciales = variables_base.copy()

                tablero_final = self.simplex_paso_paso(tablero_inicial, columnas, variables_base)

                # Análisis final e impresión de múltiples óptimos
                self.analizar_y_mostrar_resultados(tablero_final, columnas, variables_base, len(vector_c), es_max)

                ###
                self.ejecutar_analisis_sensibilidad(tablero_final, columnas, variables_base, variables_base_iniciales, vector_c, c_calc, matriz_A, vector_b, vector_signos, es_max)


                # --- MÉTODO GRÁFICO (Si aplica) ---
                if len(vector_c) == 2:
                    print("\n" + "="*55)
                    print("📊 MÉTODO GRÁFICO (2D)")
                    print("="*55)
                    self.ejecutar_metodo_grafico(vector_c, matriz_A, vector_b, vector_signos, es_max)

            except Exception as e:
                print(f"❌ Fallo crítico durante la ejecución: {e}")

    # =========================================================================
    # --- ENGINE SIMPLEX Y GRAN M ---
    # =========================================================================
    def init_tablero_completo(self, c, A, b, signos):
        num_restr, num_vars = A.shape
        num_S = np.sum(signos == '<=')
        num_E = np.sum(signos == '>=')
        num_A = np.sum((signos == '>=') | (signos == '='))

        total_vars = num_vars + num_S + num_E + num_A
        tablero = np.zeros((num_restr + 1, total_vars + 1))

        tablero[:num_restr, :num_vars] = A
        tablero[:num_restr, -1] = b
        tablero[-1, :num_vars] = -c

        columnas = [f"X{i+1}" for i in range(num_vars)]
        base = []

        col_actual = num_vars
        idx_S, idx_E, idx_A = 1, 1, 1
        M = 1000000.0  # Gran M
        filas_artificiales = []

        for i in range(num_restr):
            
            if signos[i] == '<=':
                tablero[i, col_actual] = 1
                columnas.append(f"S{idx_S}")
                base.append(f"S{idx_S}")
                col_actual += 1; idx_S += 1
            
            elif signos[i] == '>=':
                tablero[i, col_actual] = -1
                columnas.append(f"E{idx_E}")
                col_actual += 1; idx_E += 1

                tablero[i, col_actual] = 1
                tablero[-1, col_actual] = M
                columnas.append(f"A{idx_A}")
                base.append(f"A{idx_A}")
                filas_artificiales.append(i)
                col_actual += 1; idx_A += 1
            
            elif signos[i] == '=':
                tablero[i, col_actual] = 1
                tablero[-1, col_actual] = M
                columnas.append(f"A{idx_A}")
                base.append(f"A{idx_A}")
                filas_artificiales.append(i)
                col_actual += 1; idx_A += 1

        columnas.append("RHS")
        # Volver 0 la fila Z de las variables base artificiales
        for i in filas_artificiales:
            tablero[-1, :] -= M * tablero[i, :]

        return tablero, columnas, base

    def simplex_paso_paso(self, tablero, columnas, vars_base):
        iteracion = 0
        while True:
            df = pd.DataFrame(np.round(tablero, 3), columns=columnas, index=vars_base + ["Z"])
            print(f"\n--- TABLERO ITERACIÓN {iteracion} ---")
            display(df)

            #resaltar pivotes
            if iteracion > 0:
                display(df.style
                    .apply(lambda col: [
                        'background-color:#fff3cd' if col.name == _col_entra else ''
                        for _ in col], axis=0)          # columna entrante: amarillo
                    .apply(lambda fila: [
                        'background-color:#d4edda' if fila.name == _fila_sale else ''
                        for _ in fila], axis=1)          # fila saliente: verde
                    .applymap(lambda v: 'background-color:#fd7e14;color:white;font-weight:bold'
                        if v == round(_pivote_val, 3) else '',
                        subset=pd.IndexSlice[_fila_sale, _col_entra])  # celda: naranja
                )
            else:
                display(df)



            # Condición de Optimalidad (Todos los Cj-Zj >= 0)
            if np.all(tablero[-1, :-1] >= -1e-5):
                break

            piv_c = np.argmin(tablero[-1, :-1])
            rhs = tablero[:-1, -1]
            piv_col_val = tablero[:-1, piv_c]
            ratios = np.divide(rhs, piv_col_val, out=np.full_like(rhs, np.inf), where=piv_col_val > 1e-8)

            if np.all(np.isinf(ratios)):
                raise ValueError("Problema NO ACOTADO (La región factible es infinita en dirección de Z).")

            piv_f = np.argmin(ratios)


            #columna de razon
            ratios_display = []
            for i, r in enumerate(ratios):
                ratios_display.append("∞" if np.isinf(r) else f"{r:.3f}")
            ratios_display.append("—")          # fila Z no tiene razón
            df["Razón (b/a)"] = ratios_display

            #Guardar info del pivote para siguiente iteracion
            _col_entra = columnas[piv_c]
            _fila_sale = vars_base[piv_f]
            _pivote_val = tablero[piv_f, piv_c]

            print(f"➜ Transición: Entra {columnas[piv_c]} y sale {vars_base[piv_f]}. Valor Pivote: {tablero[piv_f, piv_c]:.2f}")

            vars_base[piv_f] = columnas[piv_c]
            tablero[piv_f, :] /= tablero[piv_f, piv_c]

            for f in range(tablero.shape[0]):
                if f != piv_f:
                    tablero[f, :] -= tablero[f, piv_c] * tablero[piv_f, :]
            iteracion += 1

        return tablero

    def analizar_y_mostrar_resultados(self, tablero, columnas, vars_base, n_vars, es_max):
        # 1. VERIFICAR INFACTIBILIDAD
        for i, var in enumerate(vars_base):
            if var.startswith('A') and tablero[i, -1] > 1e-5:
                print("\n❌ ATENCIÓN: EL PROBLEMA ES INFACTIBLE.")
                print("Existe una contradicción en las restricciones (Variable Artificial en la base > 0).")
                return

        # 2. EXTRAER SOLUCIÓN ÓPTIMA PRINCIPAL
        print("\n🏆 VALORES ÓPTIMOS DE LAS VARIABLES ENCONTRADOS:")
        valores_primarios = []
        for idx_var in range(n_vars):
            columna = tablero[:, idx_var]
            if np.sum(np.isclose(columna, 1, atol=1e-5)) == 1 and np.sum(np.isclose(columna, 0, atol=1e-5)) == len(columna) - 1:
                fila_uno = np.where(np.isclose(columna, 1, atol=1e-5))[0][0]
                valor = tablero[fila_uno, -1]
            else:
                valor = 0.0
            valores_primarios.append(valor)
            print(f"   Variable X{idx_var+1} = {valor:.2f}")

        z_optima = tablero[-1, -1] if es_max else -tablero[-1, -1]
        print(f"💰 Utilidad/Costo Óptimo (Z) = {z_optima:.2f}")

        # =================================================================
        # MEJORA 2: DETECCIÓN DE MÚLTIPLES SOLUCIONES ÓPTIMAS
        # =================================================================
        optimos_encontrados = False

        # Escaneamos todas las columnas buscando un '0' en la fila Z de una variable no básica
        for j in range(tablero.shape[1] - 1):
            if columnas[j].startswith('A'): continue # Ignoramos las artificiales

            columna = tablero[:, j]
            es_basica = (np.sum(np.isclose(columna, 1, atol=1e-5)) == 1 and np.sum(np.isclose(columna, 0, atol=1e-5)) == len(columna) - 1)

            # Si NO es básica, pero tiene un costo reducido de cero (0):
            if not es_basica and np.isclose(tablero[-1, j], 0, atol=1e-5):

                # Simulamos la entrada de esta variable iterando el tablero una vez más
                rhs = tablero[:-1, -1]
                piv_col_val = tablero[:-1, j]
                ratios = np.divide(rhs, piv_col_val, out=np.full_like(rhs, np.inf), where=piv_col_val > 1e-8)

                if not np.all(np.isinf(ratios)): # Si el problema no es ilimitado
                    piv_f = np.argmin(ratios)
                    tab_copy = tablero.copy()

                    # Pivoteo en la copia
                    tab_copy[piv_f, :] /= tab_copy[piv_f, j]
                    for f in range(tab_copy.shape[0]):
                        if f != piv_f:
                            tab_copy[f, :] -= tab_copy[f, j] * tab_copy[piv_f, :]

                    # Extraer los nuevos valores de X de la solución alternativa
                    valores_alt = []
                    for idx_var in range(n_vars):
                        c_temp = tab_copy[:, idx_var]
                        if np.sum(np.isclose(c_temp, 1, atol=1e-5)) == 1 and np.sum(np.isclose(c_temp, 0, atol=1e-5)) == len(c_temp) - 1:
                            fila_uno = np.where(np.isclose(c_temp, 1, atol=1e-5))[0][0]
                            valores_alt.append(tab_copy[fila_uno, -1])
                        else:
                            valores_alt.append(0.0)

                    # Comprobar si el nuevo vértice es geométricamente diferente al primero (evitar degeneración)
                    if not np.allclose(valores_primarios, valores_alt, atol=1e-5):
                        if not optimos_encontrados:
                            print("\n⚠️ ¡ATENCIÓN: MÚLTIPLES SOLUCIONES ÓPTIMAS DETECTADAS! ⚠️")
                            print("Existen infinitas combinaciones que arrojan el mismo Z óptimo. Aquí tienes otro vértice válido:")
                            optimos_encontrados = True

                        print(f"➜ Solución Alternativa (Entrando {columnas[j]} a la base):")
                        for idx_var, val_alt in enumerate(valores_alt):
                            print(f"   Variable X{idx_var+1} = {val_alt:.2f}")



# =========================================================================
    # --- SPRINT 3: ENGINE DE ANÁLISIS DE SENSIBILIDAD (ESTILO EXCEL) ---
    # =========================================================================
    def ejecutar_analisis_sensibilidad(self, tablero_final, columnas, variables_base, variables_base_iniciales, vector_c, c_calc, matriz_A, vector_b, vector_signos, es_max):
        print("\n" + "="*55)
        print("📈 REPORTES DE SENSIBILIDAD (ESTILO EXCEL)")
        print("="*55)
        
        num_restr, n_vars = matriz_A.shape
        M = 1000000.0

        #1. Extracción de la matriz inversa optima B^-1 a partir del tablero final y verificar que existe la columna
        B_inv = np.zeros((num_restr, num_restr))
        for i in range(num_restr):

            var_ini = variables_base_iniciales[i]

            if var_ini not in columnas:
                B_inv[i, i] = 1.0
            else:
                col_idx = columnas.index(var_ini)
                B_inv[:, i] = tablero_final[:-1, col_idx]


        #2. Calculo de los valores optimos)
        valores_primarios = []
        for j in range(n_vars):
            col = tablero_final[:-1, j]
            if f"X{j+1}" in variables_base:
                fila_idx = variables_base.index(f"X{j+1}")
                valores_primarios.append(tablero_final[fila_idx, -1])
            else:
                valores_primarios.append(0.0)
        valores_primarios = np.array(valores_primarios)

        #3. Reporte de celdas de variables
        filas_variables = []
        for j in range(n_vars):
            var_name = f"X{j+1}"
            val_final = valores_primarios[j]
            costo_reducido = tablero_final[-1, j]
            coef_original = vector_c[j]

            # Encontrar los rangos para coeficientes de la F.O.
            if var_name not in variables_base:
                # Variable No Básica
                if es_max:
                    aumento_p = costo_reducido
                    disminucion_p = np.inf
                else:
                    aumento_p = np.inf
                    disminucion_p = costo_reducido
            else:
                # Variable Básica: Analizar impacto sobre las columnas no básicas
                fila_k = variables_base.index(var_name)
                aumento_max_board = np.inf
                disminucion_max_board = np.inf

                for m in range(tablero_final.shape[1] - 1):
                    col_m_name = columnas[m]
                    columna_m = tablero_final[:-1, m]
                    es_m_basica = (col_m_name in variables_base)
                    
                    if es_m_basica or col_m_name.startswith('A'): 
                        continue

                    coef_fila_k = tablero_final[fila_k, m]
                    costo_red_m = tablero_final[-1, m]

                    if coef_fila_k > 1e-8:
                        val = costo_red_m / coef_fila_k
                        if val < disminucion_max_board: disminucion_max_board = val
                    elif coef_fila_k < -1e-8:
                        val = -costo_red_m / coef_fila_k
                        if val < aumento_max_board: aumento_max_board = val

                if es_max:
                    aumento_p = aumento_max_board
                    disminucion_p = disminucion_max_board
                else:
                    aumento_p = disminucion_max_board
                    disminucion_p = aumento_max_board

            filas_variables.append([
                var_name, 
                np.round(val_final, 4), 
                np.round(costo_reducido if es_max else -costo_reducido, 4), 
                np.round(coef_original, 4), 
                np.round(aumento_p, 4) if aumento_p != np.inf else "Infinito", 
                np.round(disminucion_p, 4) if disminucion_p != np.inf else "Infinito"
            ])

        df_variables = pd.DataFrame(filas_variables, columns=[
            'Variable', 'Valor Final', 'Costo Reducido', 'Coeficiente Original', 'Aumento Permisible', 'Disminución Permisible'
        ])
        print("\n🔹 REPORTE DE VARIABLES:")
        display(df_variables)

        #4. Reporte de restricciones (Precios sombra & RHS)
        # Cálculo unificado de Precios Sombra usando: C_B * B^-1
        C_B = []
        for var in variables_base:
            if var.startswith('X'):
                idx = int(var[1:]) - 1
                C_B.append(vector_c[idx]) #se calcula con vector_c para no doblar negacion
            else:
                C_B.append(0.0)
        C_B = np.array(C_B)
        
        precios_sombra = C_B @ B_inv
        if not es_max:
            precios_sombra = -precios_sombra # Invertir para escala real de Minimización

        uso_recursos = matriz_A @ valores_primarios

        filas_restricciones = []
        for k in range(num_restr):
            restr_name = f"R{k+1}"
            val_final = uso_recursos[k]
            p_sombra = precios_sombra[k]
            rhs_original = vector_b[k]

            # Calcular rangos del lado derecho (RHS) usando la columna k de B^-1
            v_col_k = B_inv[:, k]
            x_B = tablero_final[:-1, -1]
            
            aumento_rhs = np.inf
            disminucion_rhs = np.inf

            for i in range(num_restr):
                v_i = v_col_k[i]
                x_Bi = x_B[i]
                if v_i > 1e-8:
                    val = x_Bi / v_i
                    if val < disminucion_rhs: disminucion_rhs = val
                elif v_i < -1e-8:
                    val = -x_Bi / v_i
                    if val < aumento_rhs: aumento_rhs = val

            filas_restricciones.append([
                restr_name, 
                np.round(val_final, 4), 
                np.round(p_sombra, 4), 
                np.round(rhs_original, 4), 
                np.round(aumento_rhs, 4) if aumento_rhs != np.inf else "Infinito", 
                np.round(disminucion_rhs, 4) if disminucion_rhs != np.inf else "Infinito"
            ])

        df_restricciones = pd.DataFrame(filas_restricciones, columns=[
            'Restricción', 'Valor Final (Uso)', 'Precio Sombra', 'RHS Original', 'Aumento Permisible', 'Disminución Permisible'
        ])
        print("\n🔸 REPORTE DE RESTRICCIONES:")
        display(df_restricciones)




    # =========================================================================
    # --- ENGINE GEOMÉTRICO (GRÁFICA 2D) ---
    # =========================================================================
    def ejecutar_metodo_grafico(self, c, A, b, signos, es_max):
        try:
            vertices = self.hallar_vertices_region_factible(A, b, signos)
            if vertices.size == 0:
                print("❌ Conflicto geométrico: Región Factible Vacía (Infactible).")
                return

            if vertices.shape[0] < 3:
                print("⚠️ Región factible con menos de 3 vértices — no se puede graficar el polígono.")
                self.graficar_solucion(vertices, vertices[0] if len(vertices) else np.array([0,0]),
                                    0, c, A, b, signos, es_max)
                return

            """valores_z = vertices @ c
            piv_opt = np.argmax(valores_z) if es_max else np.argmin(valores_z)

            self.graficar_solucion(vertices, vertices[piv_opt], valores_z[piv_opt], c, A, b, signos, es_max)
        """
        except Exception as e:
            print(f"⚠️ Error al renderizar gráfica: {e}")
        
        try:
            hull = ConvexHull(vertices)
            vertices_hull = vertices[hull.vertices]
        except Exception:
            print("⚠️ Región factible no acotada: se grafican restricciones sin relleno.")
            vertices_hull = vertices

        valores_z = vertices_hull @ c
        piv_opt   = np.argmax(valores_z) if es_max else np.argmin(valores_z)
        self.graficar_solucion(vertices_hull, vertices_hull[piv_opt], valores_z[piv_opt], c, A, b, signos, es_max)



    def hallar_vertices_region_factible(self, A, b, signos):
        n_restr = A.shape[0]
        lineas_A = np.vstack([A, [1, 0], [0, 1]])
        lineas_b = np.hstack([b, 0, 0])
        puntos = []

        for i in range(len(lineas_b)):
            for j in range(i + 1, len(lineas_b)):
                M = np.vstack([lineas_A[i], lineas_A[j]])
                rhs = np.array([lineas_b[i], lineas_b[j]])
                try:
                    p = np.linalg.solve(M, rhs)
                    Tol = 1e-6
                    factible = True
                    for k in range(n_restr):
                        calc = A[k] @ p
                        if signos[k] == '<=' and calc > b[k] + Tol: factible = False; break
                        if signos[k] == '>=' and calc < b[k] - Tol: factible = False; break
                        if signos[k] == '=' and not np.isclose(calc, b[k], atol=Tol): factible = False; break

                    if p[0] < -Tol or p[1] < -Tol: factible = False
                    if factible: puntos.append(p)
                except np.linalg.LinAlgError:
                    pass

        return np.unique(np.array(puntos), axis=0) if puntos else np.array([])

    def graficar_solucion(self, vertices, p_opt, z_opt, c, A, b, signos, es_max):
        plt.figure(figsize=(9, 8))
        max_v = np.max(vertices) * 1.3 if vertices.size > 0 else 10
        xlim = (0, max(max_v, 10)); ylim = (0, max(max_v, 10))
        x_vals = np.linspace(xlim[0], xlim[1], 400)

        cols = ['blue', 'green', 'purple', 'cyan', 'darkgreen']
        for i in range(A.shape[0]):
            a1, a2 = A[i]
            color = cols[i % len(cols)]
            if a2 != 0:
                plt.plot(x_vals, (b[i] - a1 * x_vals) / a2, label=f"Restricción {i+1}", color=color, linestyle='--', alpha=0.8)
            else:
                plt.axvline(x=b[i]/a1, label=f"Restricción {i+1}", color=color, linestyle='--', alpha=0.8)

        plt.fill(vertices[:, 0], vertices[:, 1], color='gold', alpha=0.3, label='Región Factible')

        c1, c2 = c
        if c2 != 0:
            plt.plot(x_vals, (z_opt - c1 * x_vals) / c2, color='crimson', linestyle='-', linewidth=2.5, label='Recta de Isoutilidad ÓPTIMA')
        else:
            plt.axvline(x=z_opt/c1, color='crimson', linestyle='-', linewidth=2.5, label='Recta de Isoutilidad ÓPTIMA')

        plt.scatter(p_opt[0], p_opt[1], color='crimson', s=180, edgecolors='black', zorder=5)
        plt.annotate(f" ÓPTIMO\n X1={p_opt[0]:.2f}, X2={p_opt[1]:.2f}\n Z={z_opt:.2f}", p_opt, xytext=(p_opt[0]+0.3, p_opt[1]+0.3), fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5))

        plt.xlim(xlim); plt.ylim(ylim)
        plt.xlabel('X1'); plt.ylabel('X2')
        plt.title(f"Análisis Espacial del Modelo Lineal")
        plt.axhline(0, color='black'); plt.axvline(0, color='black')
        plt.grid(True, linestyle=':', alpha=0.5); plt.legend(loc='upper right')
        display(plt.gcf())
        plt.close()

# Iniciar App
app = InterfazPL()

HTML(value='<h3>Calculadora de Programación Lineal (Gran M + Múltiples Óptimos)</h3>')

HTML(value='<small><i>(Gráfico disponible solo para 2 variables)</i></small>', layout=Layout(margin='0 10px 0 …

VBox()

Output()